# AISKG Framework v3.0.0 — Complete Unified Pipeline

This notebook executes the frozen manuscript-compatible Section 1 and Section 2 workflows, the nine-configuration ablation suite, publication figures, reproducibility audits, and deterministic release packaging.

[Open this notebook in Google Colab](https://colab.research.google.com/github/romenmeitei/AISKG_Framework/blob/main/notebooks/AISKG_Framework_v3_Complete_Pipeline.ipynb)

In [1]:
#@title Run configuration
REPOSITORY_URL = "https://github.com/romenmeitei/AISKG_Framework.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
CONFIG_PATH = "configs/manuscript_frozen.yaml" #@param {type:"string"}
RUN_ID = "colab-publication-v3" #@param {type:"string"}
FORCE_RECLONE = False #@param {type:"boolean"}


In [2]:
import os
import pathlib
import shutil
import subprocess
import sys

local_override = os.environ.get("AISKG_LOCAL_REPOSITORY")
if local_override:
    repository = pathlib.Path(local_override).expanduser().resolve()
else:
    repository = pathlib.Path("/content/AISKG_Framework")
    if FORCE_RECLONE and repository.exists():
        shutil.rmtree(repository)
    if not repository.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPOSITORY_URL, str(repository)], check=True)

os.chdir(repository)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-build-isolation"], check=True)
print(f"Repository: {repository}")
print(f"Python: {sys.version.split()[0]}")


Repository: /mnt/data/AISKG_Framework_v3.0.0_GITHUB_READY
Python: 3.13.5


In [3]:
import subprocess
import sys

command = [
    sys.executable,
    "run_pipeline.py",
    "--config", CONFIG_PATH,
    "--run-id", RUN_ID,
    "--clean",
]
print("Executing:", " ".join(command))
subprocess.run(command, check=True)


Executing: /opt/pyvenv/bin/python run_pipeline.py --config configs/manuscript_frozen.yaml --run-id colab-publication-v3 --clean


2026-08-05 16:13:47,810 | INFO | Starting AISKG 3.0.0 run colab-publication-v3
2026-08-05 16:13:47,810 | INFO | Section 1: extracting AISKG_Section1_Inputs_v1.0.0.zip
2026-08-05 16:13:47,901 | INFO | Section 1: executing frozen literature-to-extraction pipeline


2026-08-05 16:13:56,197 | INFO | Section 2: extracting Mushroom_KG_Reproducibility_Inputs_v2_from_upstream.zip
2026-08-05 16:13:56,232 | INFO | Section 2: executing frozen post-extraction pipeline


2026-08-05 16:14:14,672 | INFO | Ablation: rebuilding nine frozen-corpus variants and comparison outputs


2026-08-05 16:14:17,413 | INFO | Creating deterministic release archive: /mnt/data/AISKG_Framework_v3.0.0_GITHUB_READY/outputs/colab-publication-v3/AISKG_Framework_v3.0.0_Release.zip


{
  "run_dir": "/mnt/data/AISKG_Framework_v3.0.0_GITHUB_READY/outputs/colab-publication-v3",
  "release_zip": "/mnt/data/AISKG_Framework_v3.0.0_GITHUB_READY/outputs/colab-publication-v3/AISKG_Framework_v3.0.0_Release.zip",
  "release_archive": "/mnt/data/AISKG_Framework_v3.0.0_GITHUB_READY/outputs/colab-publication-v3/AISKG_Framework_v3.0.0_Release.zip"
}


CompletedProcess(args=['/opt/pyvenv/bin/python', 'run_pipeline.py', '--config', 'configs/manuscript_frozen.yaml', '--run-id', 'colab-publication-v3', '--clean'], returncode=0)

In [4]:
from pathlib import Path
import pandas as pd
from aiskg.reproducibility import verify_run

run_dir = Path("outputs") / RUN_ID
verification = verify_run(run_dir)
audit = pd.read_csv(run_dir / "outputs" / "reproducibility_audit.csv")
ablation = pd.read_csv(run_dir / "outputs" / "extensions" / "ablation" / "ablation_summary.csv")

print((run_dir / "PIPELINE_SUCCESS.txt").read_text().strip())
print(f"Audit: {(audit.status == 'PASS').sum()}/{len(audit)} checks passed")
print("Verification:", verification)
display(ablation[[
    "label", "entity_f1", "relation_f1", "exact_triple_accuracy",
    "nodes", "edges", "modularity", "pathway_count"
]])
release_zip = run_dir / "AISKG_Framework_v3.0.0_Release.zip"
print("Release archive:", release_zip)


SUCCESS
Audit: 285/285 checks passed
Verification: {'files_checked': 208, 'success_markers': 1}


,label,entity_f1,relation_f1,exact_triple_accuracy,nodes,edges,modularity,pathway_count
0,Full framework,0.894928,0.943396,0.892857,38,77,0.330395,52
1,Without canonical normalization,0.659498,0.509434,0.482143,67,135,0.466787,85
2,Without ontology type constraints,0.894928,0.283951,0.150943,57,258,0.337225,44
3,Without semantic quality filters,0.894928,0.943396,0.892857,38,78,0.324787,105
4,Without outcome-aware refinement,0.894928,0.943396,0.892857,40,86,0.281716,95
5,Support >=1,0.894928,0.943396,0.892857,62,170,0.331108,213
6,Support >=2,0.894928,0.943396,0.892857,38,77,0.330395,52
7,Support >=3,0.894928,0.943396,0.892857,34,56,0.326579,22
8,Support >=5,0.894928,0.943396,0.892857,25,39,0.325980,14


Release archive: outputs/colab-publication-v3/AISKG_Framework_v3.0.0_Release.zip


In [5]:
# Download the deterministic release when running in Google Colab.
try:
    from google.colab import files
    files.download(str(release_zip))
except ImportError:
    print("Not running inside Google Colab; release remains at:", release_zip)


Not running inside Google Colab; release remains at: outputs/colab-publication-v3/AISKG_Framework_v3.0.0_Release.zip
